In [13]:
# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
print(training_dataframe['Sentiment'].value_counts())
print(training_dataframe['Sarcasm'].value_counts())
print(training_dataframe['variety'].value_counts())




DatasetDict({
    train: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 3747
    })
    validation: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 313
    })
    test: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 2183
    })
})
Sentiment
0.0    1907
1.0    1840
Name: count, dtype: int64
Sarcasm
0.0    3223
1.0     524
Name: count, dtype: int64
variety
en-IN    1399
en-UK    1203
en-AU    1145
Name: count, dtype: int64


In [11]:
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(training_dataset['text'])
X_test = vectorizer.transform(dataset['test'].to_pandas()['text'])

# Sentiment
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataset['Sentiment'])
sentiment_predictions = lr_sentiment.predict(X_test)

test_dataframe = dataset['test'].to_pandas()
print("Sentiment metrics", classification_report(test_dataset['Sentiment'], sentiment_predictions))

# Sarcasm
lr_sarcasm = LogisticRegression(max_iter=1000)
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print("Sarcasm metrics", classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))

Sentiment metrics               precision    recall  f1-score   support

         0.0       0.81      0.88      0.84      1117
         1.0       0.86      0.79      0.82      1066

    accuracy                           0.83      2183
   macro avg       0.83      0.83      0.83      2183
weighted avg       0.83      0.83      0.83      2183

Sarcasm metrics               precision    recall  f1-score   support

         0.0       0.86      1.00      0.92      1878
         1.0       0.00      0.00      0.00       305

    accuracy                           0.86      2183
   macro avg       0.43      0.50      0.46      2183
weighted avg       0.74      0.86      0.80      2183



These results show that accuracy is the wrong metric. For sarcasm the dataset is highly imbalanced. The 0.86 accuracy is misleading. The precision and recall for sarcasm is 0.00. The macro score of 0.46 is more useful.

In [16]:
# Balanced class weight
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataset['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print(classification_report(test_dataset['Sarcasm'], sarcasm_predictions))

              precision    recall  f1-score   support

         0.0       0.93      0.77      0.85      1878
         1.0       0.32      0.65      0.43       305

    accuracy                           0.76      2183
   macro avg       0.62      0.71      0.64      2183
weighted avg       0.85      0.76      0.79      2183



The model is now detecting sarcasm. The accuracy is lower but the macro and weighted averages are higher.

In [22]:
from sklearn.svm import LinearSVC

# Sentiment
svm_sentiment = LinearSVC(max_iter=10000)
svm_sentiment.fit(X_train, training_dataframe['Sentiment'])
print("SVM Sentiment metrics:")
print(classification_report(test_dataframe['Sentiment'], svm_sentiment.predict(X_test)))
# Sarcasm
svm_sarcasm = LinearSVC(max_iter=10000, class_weight='balanced')
svm_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
print("SVM Sarcasm metrics:")
print(classification_report(test_dataframe['Sarcasm'], svm_sarcasm.predict(X_test)))

SVM Sentiment metrics:
              precision    recall  f1-score   support

         0.0       0.83      0.85      0.84      1117
         1.0       0.84      0.82      0.83      1066

    accuracy                           0.84      2183
   macro avg       0.84      0.83      0.83      2183
weighted avg       0.84      0.84      0.83      2183

SVM Sarcasm metrics:
              precision    recall  f1-score   support

         0.0       0.89      0.89      0.89      1878
         1.0       0.32      0.33      0.32       305

    accuracy                           0.81      2183
   macro avg       0.60      0.61      0.61      2183
weighted avg       0.81      0.81      0.81      2183

